# **Virtual Try-On using IP-Adapter Inpainting**

This notebook implements a virtual try-on system using Stable Diffusion XL with IP-Adapter for inpainting.
The system allows users to virtually try on clothing items by combining person images with clothing images.

**Overview**
- **Model**: Stable Diffusion XL 1.0 Inpainting
- **Technique**: IP-Adapter for image-guided generation
- **Segmentation**: Automatic body segmentation for mask creation
- **API**: FastAPI server with ngrok tunnel for external access

**Requirements**
- GPU with CUDA support (T4 or better recommended)
- Python 3.8+
- Sufficient disk space for model downloads (~10GB)

**Step 1: Install Required Libraries**

Install core dependencies for diffusion models and image processing.

In [ ]:
# Install diffusers library for Stable Diffusion models
# Install accelerate for optimized model loading and inference
!pip install diffusers accelerate

**Step 2: Import Core Libraries**

Import necessary modules for the diffusion pipeline and image handling.

In [ ]:
from diffusers import AutoPipelineForInpainting, AutoencoderKL
from diffusers.utils import load_image
import torch

**Step 3: Initialize the Pipeline**

Load the SDXL inpainting model with IP-Adapter for image-guided generation.
This step downloads the models (~10GB) and may take several minutes.

In [ ]:
# Load optimized VAE for better image quality with fp16 precision
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16
)

# Initialize the main inpainting pipeline
pipeline = AutoPipelineForInpainting.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    vae=vae,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")

# Load IP-Adapter weights for image-guided generation
pipeline.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="sdxl_models",
    weight_name="ip-adapter_sdxl.bin",
    low_cpu_mem_usage=True
)

**Step 4: Setup Body Segmentation Tool**

Install the body segmentation tool to automatically create masks for the person's body.
This eliminates the need for manual mask creation.

In [ ]:
# Clone the Segment-Body repository
!git clone https://github.com/TonyAssi/Segment-Body.git
%cd /content/Segment-Body

# Install segmentation dependencies
!pip install -r requirements.txt

# Copy the segmentation module to the parent directory
!cp ./SegBody.py ..
%cd ..

In [ ]:
# Import the body segmentation function
from SegBody import segment_body

# Example usage:
# seg_image, mask_image = segment_body(image, face=False)
# mask_image.resize((512, 512))

**Step 5: Configure IP-Adapter Scale**

Set the influence strength of the reference clothing image.
Range: 0.0 (no influence) to 1.0 (maximum influence)

In [ ]:
# Set IP-Adapter scale to maximum for strongest clothing reference
pipeline.set_ip_adapter_scale(1.0)

# **Step 6: Define Virtual Try-On Function**

Main function that orchestrates the virtual try-on process:
1. Segments the person's body to create a mask
2. Applies the clothing image using IP-Adapter
3. Generates the final result using inpainting

In [ ]:
def virtual_try_on(
    img,
    clothing,
    prompt,
    negative_prompt,
    ip_scale=1.0,
    strength=0.99,
    guidance_scale=7.5,
    steps=100
):
    """
    Perform virtual try-on by combining a person image with a clothing image.

    Parameters:
    -----------
    img : PIL.Image
        Image of the person
    clothing : PIL.Image
        Image of the clothing item
    prompt : str
        Positive prompt describing desired output
    negative_prompt : str
        Negative prompt to avoid unwanted features
    ip_scale : float, default=1.0
        IP-Adapter influence (0.0-1.0)
    strength : float, default=0.99
        Denoising strength (0.0-1.0)
    guidance_scale : float, default=7.5
        Classifier-free guidance scale
    steps : int, default=100
        Number of diffusion steps

    Returns:
    --------
    tuple : (mask_image, result_image)
        Generated mask and final try-on result
    """
    # Generate body segmentation mask (excluding face)
    _, mask_img = segment_body(img, face=False)

    # Configure IP-Adapter strength
    pipeline.set_ip_adapter_scale(ip_scale)

    # Generate the try-on result
    images = pipeline(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=img,
        mask_image=mask_img,
        ip_adapter_image=clothing,
        strength=strength,
        guidance_scale=guidance_scale,
        num_inference_steps=steps,
    ).images

    return mask_img, images[0]

**Step 7: Load Sample Images**

Load example images for testing the virtual try-on system.

In [ ]:
# Load sample person image
image = load_image(
    'https://cdn-uploads.huggingface.co/production/uploads/648a824a8ca6cf9857d1349c/jpFBKqYB3BtAW26jCGJKL.jpeg'
).convert("RGB")

# Load sample clothing image
ip_image = load_image(
    'https://cdn-uploads.huggingface.co/production/uploads/648a824a8ca6cf9857d1349c/NL6mAYJTuylw373ae3g-Z.jpeg'
).convert("RGB")

In [ ]:
# Import display utility for showing results in notebook
from IPython.display import display

# **Step 8: Interactive Try-On (Optional)**

Use this cell to test the virtual try-on with custom images.
Uncomment and modify the URLs to try different combinations.

In [ ]:
# Example usage (uncomment to use):
# image = load_image(input("Enter the person's image URL: ")).convert("RGB")
# ip_image = load_image(input("Enter the clothing URL: ")).convert("RGB")

# mask_image, result = virtual_try_on(
#     img=image,
#     clothing=ip_image,
#     prompt="photorealistic, perfect body, beautiful skin, realistic skin, natural skin",
#     negative_prompt="ugly, bad quality, bad anatomy, deformed body, deformed hands, deformed feet, deformed face, deformed clothing, deformed skin, bad skin, leggings, tights, stockings"
# )

# display(mask_image)
# display(result)

## **Step 9: Setup API Server**

Install dependencies for creating a FastAPI server with external access via ngrok.

In [ ]:
# Install web framework and server dependencies
!pip install fastapi uvicorn pydantic

# Install utilities for HTTP requests and tunneling
!pip install requests pyngrok nest-asyncio

# **Step 10: Launch API Server**

Create and run a FastAPI server that exposes the virtual try-on functionality via HTTP endpoints.
The server uses ngrok to create a public URL accessible from anywhere.

### Endpoints:
- `GET /` - Health check endpoint
- `POST /try-on` - Main virtual try-on endpoint (accepts file uploads)

### Usage:
1. Run this cell to start the server
2. Copy the public URL that appears in the output
3. Use the URL with your frontend or API client
4. Access interactive API docs at `{public_url}/docs`

In [ ]:
# Import Dependencies
from fastapi import FastAPI, HTTPException, File, UploadFile, Form
from pydantic import BaseModel
from typing import Optional
import uvicorn
from pyngrok import ngrok
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio
import base64
from io import BytesIO
from PIL import Image
import traceback
import os

# Initialize FastAPI Application
app = FastAPI(title="Virtual Try-On API", version="1.0.0")

# Configure CORS to allow requests from any origin
# Required for browser extensions and web applications
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

#Helper Functions

def image_to_base64(image: Image.Image) -> str:
    """
    Convert PIL Image to base64 encoded string.

    Parameters:
    -----------
    image : PIL.Image.Image
        Input image to convert

    Returns:
    --------
    str : Base64 encoded image with data URI prefix
    """
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode()
    return f"data:image/png;base64,{img_str}"


def read_image_from_bytes(file_bytes: bytes) -> Image.Image:
    """
    Read and validate image from uploaded file bytes.

    Parameters:
    -----------
    file_bytes : bytes
        Raw bytes from uploaded file

    Returns:
    --------
    PIL.Image.Image : Loaded and validated RGB image

    Raises:
    -------
    HTTPException : If image cannot be loaded or is invalid
    """
    try:
        return Image.open(BytesIO(file_bytes)).convert("RGB")
    except Exception as e:
        raise HTTPException(
            status_code=400,
            detail=f"Invalid image file provided. Error: {e}"
        )


# API Endpoints

@app.get("/")
async def root():
    """
    Health check endpoint.

    Returns:
    --------
    dict : Server status information
    """
    return {
        "status": "running",
        "message": "Virtual Try-On API is ready!"
    }


@app.post("/try-on")
async def try_on_endpoint(
    person_image: UploadFile = File(...),
    cloth_image: UploadFile = File(...),
    steps: int = Form(30)
):
    """
    Perform virtual try-on using uploaded images.

    Parameters:
    -----------
    person_image : UploadFile
        Image of the person (JPEG/PNG)
    cloth_image : UploadFile
        Image of the clothing item (JPEG/PNG)
    steps : int, default=30
        Number of diffusion steps (higher = better quality, slower)

    Returns:
    --------
    dict : JSON response containing:
        - success (bool): Operation success status
        - result_image_base64 (str): Base64 encoded result image
        - processing_time (float): Time taken in seconds
        - message (str): Status message
    """
    import time
    start_time = time.time()

    try:
        print("Processing uploaded images...")

        # Load images from uploaded file bytes
        person_img = read_image_from_bytes(await person_image.read())
        cloth_img = read_image_from_bytes(await cloth_image.read())

        print("Running virtual try-on...")

        # Execute virtual try-on
        mask_image, result_image = virtual_try_on(
            img=person_img,
            clothing=cloth_img,
            prompt="photorealistic, perfect body, beautiful skin, realistic skin, natural skin",
            negative_prompt="ugly, bad quality, bad anatomy, deformed",
            steps=steps
        )

        print("Converting result to base64...")
        result_base64 = image_to_base64(result_image)

        processing_time = time.time() - start_time
        print(f"Complete in {processing_time:.2f}s")

        return {
            "success": True,
            "result_image_base64": result_base64,
            "processing_time": round(processing_time, 2),
            "message": "Try-on successful!"
        }

    except Exception as e:
        print(f"Error during try-on: {e}")
        traceback.print_exc()

        return {
            "success": False,
            "result_image_base64": "",
            "processing_time": 0,
            "message": f"Backend Error: {str(e)}"
        }


#Server Startup

# Set ngrok authentication token
# IMPORTANT: Replace with your own token from https://dashboard.ngrok.com
os.environ["NGROK_AUTHTOKEN"]

# Terminate any existing ngrok tunnels
ngrok.kill()

# Create public tunnel to local server
public_url = ngrok.connect(8000)

# Display server information
print("\n" + "="*60)
print("SERVER STARTED!")
print("="*60)
print(f"Public URL: {public_url}")
print(f"API Documentation: {public_url}/docs")
print("="*60 + "\n")

# Enable asyncio event loop nesting for Colab compatibility
nest_asyncio.apply()

# Start the server
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()